In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score

In [68]:
df=pd.read_csv("checking_account_main.csv")
df

,date,transaction_id,description,category,type,amount,balance
0,2022-01-01,TX00001,Daily Sales Deposit,Sales Revenue,Credit,802,15802
1,2022-01-02,TX00002,Daily Sales Deposit,Sales Revenue,Credit,970,16772
2,2022-01-03,TX00003,Daily Sales Deposit,Sales Revenue,Credit,888,17660
3,2022-01-04,TX00004,Daily Sales Deposit,Sales Revenue,Credit,821,18481
4,2022-01-04,TX00005,Utility Bill Payment,Operating Expense,Debit,430,18051
...,...,...,...,...,...,...,...
872,2023-12-28,TX00873,Daily Sales Deposit,Sales Revenue,Credit,1144,551634
873,2023-12-28,TX00874,Bakery Payment,COGS,Debit,306,551328
874,2023-12-29,TX00875,Daily Sales Deposit,Sales Revenue,Credit,1090,552418
875,2023-12-30,TX00876,Daily Sales Deposit,Sales Revenue,Credit,1049,553467


In [69]:
revenue_df = df[df['category'] == 'Sales Revenue']

In [71]:
daily_revenue = revenue_df.groupby('date')['amount'].sum()

In [72]:
daily_revenue

date
2022-01-01     802
2022-01-02     970
2022-01-03     888
2022-01-04     821
2022-01-05     787
              ... 
2023-12-27     830
2023-12-28    1144
2023-12-29    1090
2023-12-30    1049
2023-12-31    1141
Name: amount, Length: 730, dtype: int64

In [70]:
expense_df = df[df['type'] == 'Debit']

In [73]:
daily_expense = expense_df.groupby('date')['amount'].sum()

In [74]:
df_payroll = pd.read_csv("gusto_payroll.csv")

In [75]:
df_payroll

,pay_date,employee_id,employee_name,role,type,amount,account
0,2022-02-15,E001,Alice Smith,Barista,Employee Pay,1865,Checking Secondary
1,2022-02-15,E002,Ben Lee,Barista,Employee Pay,1731,Checking Secondary
2,2022-02-15,E003,Carla Gomez,Manager,Employee Pay,2272,Checking Secondary
3,2022-02-15,C001,David Jones,Marketing Contractor,Contractor Pay,1134,Checking Secondary
4,2022-03-15,E001,Alice Smith,Barista,Employee Pay,1857,Checking Secondary
...,...,...,...,...,...,...,...
87,2023-11-15,C001,David Jones,Marketing Contractor,Contractor Pay,1214,Checking Secondary
88,2023-12-15,E001,Alice Smith,Barista,Employee Pay,1706,Checking Secondary
89,2023-12-15,E002,Ben Lee,Barista,Employee Pay,1729,Checking Secondary
90,2023-12-15,E003,Carla Gomez,Manager,Employee Pay,2243,Checking Secondary


In [76]:
df_payroll['pay_date'] = pd.to_datetime(df_payroll['pay_date'])
df_payroll.rename(columns={'pay_date': 'date', 'amount': 'Salaries'}, inplace=True)

In [77]:
daily_payroll = df_payroll.groupby('date')['Salaries'].sum().reset_index()

In [78]:
daily_revenue = daily_revenue.reset_index()
daily_revenue.rename(columns={'amount': 'Revenue'}, inplace=True)

daily_expense = daily_expense.reset_index()
daily_expense.rename(columns={'amount': 'Other_Expenses'}, inplace=True)

In [79]:
daily_revenue['date'] = pd.to_datetime(daily_revenue['date'])
daily_expense['date'] = pd.to_datetime(daily_expense['date'])

In [80]:
df_final = pd.merge(daily_revenue, daily_expense, on='date', how='outer')
df_final = pd.merge(df_final, daily_payroll, on='date', how='outer')

In [81]:
df_final.fillna(0, inplace=True)

,date,Revenue,Other_Expenses,Salaries
0,2022-01-01,802,0.0,0.0
1,2022-01-02,970,0.0,0.0
2,2022-01-03,888,0.0,0.0
3,2022-01-04,821,430.0,0.0
4,2022-01-05,787,0.0,0.0
...,...,...,...,...
725,2023-12-27,830,342.0,0.0
726,2023-12-28,1144,306.0,0.0
727,2023-12-29,1090,0.0,0.0
728,2023-12-30,1049,0.0,0.0


In [82]:
df_final['Total_Expenses'] = df_final['Other_Expenses'] + df_final['Salaries']
df_final['Profit'] = df_final['Revenue'] - df_final['Total_Expenses']

In [83]:
df_final = df_final.sort_values('date')

In [90]:
df_final.tail(50)

,date,Revenue,Other_Expenses,Salaries,Total_Expenses,Profit
680,2023-11-12,1077,0.0,0.0,0.0,1077.0
681,2023-11-13,745,0.0,0.0,0.0,745.0
682,2023-11-14,921,372.0,0.0,372.0,549.0
683,2023-11-15,1039,0.0,7080.0,7080.0,-6041.0
684,2023-11-16,865,0.0,0.0,0.0,865.0
685,2023-11-17,1143,216.0,0.0,216.0,927.0
686,2023-11-18,1043,0.0,0.0,0.0,1043.0
687,2023-11-19,763,0.0,0.0,0.0,763.0
688,2023-11-20,771,0.0,0.0,0.0,771.0
689,2023-11-21,1177,0.0,0.0,0.0,1177.0


In [105]:
df_final.to_csv('final_sme_data.csv', index=False)

In [106]:
df_final['Salaries'] = df_final['Salaries'].rolling(30, min_periods=1).mean()

In [107]:
df_final['Total_Expenses'] = df_final['Other_Expenses'] + df_final['Salaries']
df_final['Profit'] = df_final['Revenue'] - df_final['Total_Expenses']

In [108]:
df_final['Profit_Margin'] = df_final['Profit'] / df_final['Revenue']

In [109]:
df_final['Profit_Margin'] = df_final['Profit'] / df_final['Revenue']

In [114]:
df_final['Expense_Ratio'] = df_final['Total_Expenses'] / df_final['Revenue']

In [110]:
df_final['7d_profit'] = df_final['Profit'].rolling(7).mean()

In [111]:
threshold = df_final['Profit'].mean() - 2 * df_final['Profit'].std()

df_final['Anomaly'] = df_final['Profit'] < threshold

In [115]:
df_final['Health_Score'] = (
    (df_final['Profit_Margin'] * 0.5) +
    ((1 - df_final['Expense_Ratio']) * 0.5)
)

In [118]:
def get_health(score):
    if score > 0.6:
        return "Healthy"
    elif score > 0.3:
        return "Moderate"
    else:
        return "At Risk"

df_final['Health_Status'] = df_final['Health_Score'].apply(get_health)

In [119]:
df_final.to_csv('final_sme_data.csv', index=False)